# Topic 19a - Fallbacks / failover

Companion notebook for `fallbacks.py`. It shows ordered failover after
retries, the three fallback types, cooldowns, attribution and guards.

## 0. Import the core

In [ ]:
from fallbacks import (
    Deployment,
    ErrorKind,
    FakeClock,
    ModelGroup,
    ProviderError,
    Router,
    fallback_type_for,
)

print("imported fallbacks.py")

## 1. Model groups + a responder

A *model group* is the name a client asks for; it is backed by one or more
*deployments*. The `responder` stands in for the network.

In [ ]:
groups = [
    ModelGroup("primary", [Deployment("p1", "openai/gpt-a")]),
    ModelGroup("backup", [Deployment("b1", "anthropic/claude-b")]),
    ModelGroup("backup2", [Deployment("b2", "google/gemini-c")]),
]


def healthy(dep, tokens):
    return f"ok:{dep.id}"


print("groups:", [g.name for g in groups])

## 2. A healthy primary serves directly (no fallback)

In [ ]:
router = Router(groups, healthy, fallbacks={"primary": ["backup"]})
log = router.completion("primary")
print("served by:", log.served_model_group)
print("fallback-served:", log.served_by_fallback)
print("spend row:", log.as_row())

## 3. Fail over in order

The primary raises 429. With `num_retries=2` the primary deployment is tried
`1 + 2` times, then the chain is walked in order.

In [ ]:
calls = []


def primary_is_down(dep, tokens):
    calls.append(dep.id)
    if dep.id == "p1":
        raise ProviderError(ErrorKind.RATE_LIMIT, "429 rate limited")
    if dep.id == "b1":
        raise ProviderError(ErrorKind.SERVER, "500 upstream error")
    return f"ok:{dep.id}"


router = Router(
    groups,
    primary_is_down,
    fallbacks={"primary": ["backup", "backup2"]},
    num_retries=2,
)
log = router.completion("primary")
print("attempt order:", calls)
print("served by:", log.served_model_group)
print("attempted_fallbacks:", log.attempted_fallbacks)
print("spend row:", log.as_row())

## 4. The error type selects the fallback map

A content-policy or context-window error uses its own map, not the general one.

In [ ]:
def policy_error(dep, tokens):
    if dep.id == "p1":
        raise ProviderError(ErrorKind.CONTENT_POLICY, "content filtered by provider")
    return f"ok:{dep.id}"


router = Router(
    groups,
    policy_error,
    fallbacks={"primary": ["backup2"]},
    content_policy_fallbacks={"primary": ["backup"]},
)
log = router.completion("primary")
print("content policy -> served by:", log.served_model_group)
print("map used:", fallback_type_for(ErrorKind.CONTENT_POLICY))

In [ ]:
# Context-window fallback needs enable_pre_call_checks + a per-deployment limit.
small = [
    ModelGroup("tiny", [Deployment("t1", "openai/gpt-mini", max_input_tokens=10)]),
    ModelGroup("big", [Deployment("big1", "openai/gpt-big", max_input_tokens=100_000)]),
]

def echo(dep, tokens):
    return f"ok:{dep.id}:{tokens}t"

on = Router(small, echo, context_window_fallbacks={"tiny": ["big"]}, enable_pre_call_checks=True)
off = Router(small, echo, context_window_fallbacks={"tiny": ["big"]}, enable_pre_call_checks=False)
print("pre-call checks ON  ->", on.completion("tiny", tokens=50).served_model_group)
print("pre-call checks OFF ->", off.completion("tiny", tokens=50).served_model_group)

## 5. Retries + cooldowns (circuit-breaker-lite)

After `allowed_fails` failures in a minute, a deployment is cooled down and
skipped until the cooldown expires.

In [ ]:
clock = FakeClock()


def flaky_p1(dep, tokens):
    if dep.id == "p1":
        raise ProviderError(ErrorKind.SERVER, "500")
    return f"ok:{dep.id}"


router = Router(
    groups,
    flaky_p1,
    fallbacks={"primary": ["backup"]},
    allowed_fails=1,
    cooldown_time=30.0,
    clock=clock,
)
print("1st:", router.completion("primary").served_model_group)
print("p1 cooled down?", router._cooled_down("p1"))
print("2nd (p1 skipped):", router.completion("primary").served_model_group)
clock.advance(31)
print("after cooldown, p1 healthy again?", not router._cooled_down("p1"))

## 6. Attribution: fallback-served vs direct

In [ ]:
router = Router(groups, primary_is_down, fallbacks={"primary": ["backup2"]})
row = router.completion("primary").as_row()
print(row)
print("original_model_group != model_group -> a fallback served it")

## 7. Guards: key model access + budget

A fallback must not escalate privileges or bypass a budget.

In [ ]:
router = Router(
    groups,
    primary_is_down,
    fallbacks={"primary": ["backup"]},
    allowed_models=["primary"],  # key may not call the backup
)
log = router.completion("primary")
print("ok:", log.ok, "| attempted_fallbacks:", log.attempted_fallbacks)

expensive = [
    ModelGroup("primary", [Deployment("p1", "openai/gpt-a", cost_per_1k_tokens=0.0)]),
    ModelGroup("pricey", [Deployment("e1", "openai/gpt-4o", cost_per_1k_tokens=5.0)]),
]
router = Router(
    expensive,
    primary_is_down,
    fallbacks={"primary": ["pricey"]},
    budget_ceiling=1.0,
)
log = router.completion("primary")
print("over budget -> ok:", log.ok, "| attempted_fallbacks:", log.attempted_fallbacks)

## 8. `disable_fallbacks` on a single request

In [ ]:
router = Router(groups, primary_is_down, fallbacks={"primary": ["backup"]})
log = router.completion("primary", disable_fallbacks=True)
print("ok:", log.ok, "| served by:", log.served_model_group, "| fallbacks attempted:", log.attempted_fallbacks)

## 9. Notes

- Retry within a deployment first; fail over across groups second.
- `fallbacks` / `context_window_fallbacks` / `content_policy_fallbacks` are
  selected by the error kind; `default_fallbacks` covers unlisted groups.
- Spend logs must record `attempted_fallbacks` + `original_model_group`.
- Re-check key model access and budget on every fallback target.

Run the suite: `python -m pytest -q` in this folder.